In [ ]:
import torch

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import pandas as pd

from PIL import Image

from transformers import (

    TrOCRProcessor,

    VisionEncoderDecoderModel,

    AdamW

)

In [ ]:
MODEL_NAME = "microsoft/trocr-base-handwritten"

processor = TrOCRProcessor.from_pretrained(
    MODEL_NAME
)

model = VisionEncoderDecoderModel.from_pretrained(
    MODEL_NAME
)

In [ ]:
class OCRDataset(Dataset):

    def __init__(

            self,

            csv_file,

            processor

    ):

        self.data = pd.read_csv(
            csv_file
        )

        self.processor = processor

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        image_path = self.data.iloc[idx]["image"]

        text = self.data.iloc[idx]["text"]

        image = Image.open(
            image_path
        ).convert("RGB")

        pixel_values = self.processor(

            image,

            return_tensors="pt"

        ).pixel_values

        labels = self.processor.tokenizer(

            text,

            padding="max_length",

            max_length=512,

            truncation=True

        ).input_ids

        labels = [

            label

            if label != self.processor.tokenizer.pad_token_id

            else -100

            for label in labels

        ]

        return {

            "pixel_values":
            pixel_values.squeeze(),

            "labels":
            torch.tensor(labels)

        }

In [ ]:
train_dataset = OCRDataset(

    "../data/annotations/train.csv",

    processor

)

train_loader = DataLoader(

    train_dataset,

    batch_size=2,

    shuffle=True

)

In [ ]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

model.to(device)

In [ ]:
optimizer = AdamW(

    model.parameters(),

    lr=5e-5

)

In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        pixel_values = batch[
            "pixel_values"
        ].to(device)

        labels = batch[
            "labels"
        ].to(device)

        outputs = model(

            pixel_values=pixel_values,

            labels=labels

        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        optimizer.zero_grad()

        total_loss += loss.item()

    print(

        f"Epoch {epoch+1}"

        f" Loss = "

        f"{total_loss}"

    )

In [ ]:
torch.save(

    model.state_dict(),

    "../models/final_model/model.pth"

)

print("Model Saved")